# Lab Polaris Avancé — Jour 1

**Profil docker :** `iceberg` · **Durée :** 60 min · **Format :** Trio rôlé

---

## Qu'est-ce qu'Apache Polaris ?

Polaris est un **Iceberg REST Catalog** open-source (Apache Incubating, ex-Snowflake Open Catalog).
Il expose une API REST standard qui permet à n'importe quel moteur compatible Iceberg
(Spark, Trino, Flink, PyIceberg) de lire et écrire des tables sans se soucier du stockage sous-jacent.

```
Spark ──┐
Trino ──┤──→  Polaris REST API  ──→  MinIO (S3)
Flink ──┤          (catalog)         (data files)
PyIceberg ─┘
```

## Contexte — Mission client

> RetailCo veut exposer ses tables Iceberg à deux équipes clientes distinctes :
> - **Équipe Analytics** : accès lecture sur les tables Gold
> - **Équipe Data Science** : accès lecture/écriture sur un namespace dédié ML
>
> Votre mission : configurer Polaris en multi-tenant avec isolation stricte,
> OAuth2, et vérifier que Spark et Trino accèdent au même catalog.

In [ ]:
import requests, json, base64, time
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

POLARIS = 'http://polaris:8181'
MINIO   = 'http://minio:9000'

def polaris(method, path, data=None, token=None):
    headers = {'Content-Type': 'application/json'}
    if token:
        headers['Authorization'] = f'Bearer {token}'
    r = requests.request(method, f'{POLARIS}{path}', headers=headers, json=data)
    try:
        return r.json()
    except:
        return {'status_code': r.status_code, 'text': r.text}

# SparkSession avec Polaris
spark = (
    SparkSession.builder
    .appName('RetailCo-Polaris-Lab')
    .config('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions')
    .getOrCreate()
)

# Vérifier la connectivité Polaris
config = polaris('GET', '/api/catalog/v1/config')
print('✅ Polaris config :')
print(json.dumps(config, indent=2))
print(f'\n✅ Spark {spark.version} connecté')

---
## ⏱️ Temps 1 — Kata d'amorce (10 min)

Explorer la structure d'un catalog Polaris — namespaces, tables, propriétés.

In [ ]:
# ── Explorer le catalog existant ─────────────────────────────────────────────
print('📋 Namespaces dans le catalog Polaris :')
ns = polaris('GET', '/api/catalog/v1/namespaces')
for n in ns.get('namespaces', []):
    print(f'   {n}')

# Créer nos namespaces si nécessaire
for namespace in ['retailco', 'mlops', 'analytics']:
    resp = polaris('POST', '/api/catalog/v1/namespaces', {
        'namespace': [namespace],
        'properties': {'owner': 'formation', 'managed-by': 'polaris'}
    })
    status = 'déjà existant' if resp.get('error') else 'créé'
    print(f'   Namespace {namespace} : {status}')

# Lister les namespaces
ns_list = polaris('GET', '/api/catalog/v1/namespaces')
print('\n✅ Namespaces disponibles :')
for n in ns_list.get('namespaces', []):
    print(f'   {n}')

In [ ]:
# ── Créer une table dans Polaris et l'interroger depuis Spark et Trino ────────
spark.sql('CREATE NAMESPACE IF NOT EXISTS polaris.analytics')
spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.analytics.sales_summary
    (store_region STRING, category STRING, ca_total DOUBLE, nb_transactions BIGINT)
    USING iceberg
""")
spark.sql("""
    INSERT INTO polaris.analytics.sales_summary VALUES
    ('Île-de-France', 'Électronique', 1250000.0, 4521),
    ('PACA',          'Vêtements',     430000.0, 3210),
    ('Bretagne',      'Maison',         98000.0,  876)
""")
print('✅ Table polaris.analytics.sales_summary créée')

# Vérifier depuis PyIceberg (REST catalog)
try:
    from pyiceberg.catalog.rest import RestCatalog
    cat = RestCatalog('polaris', uri=f'{POLARIS}/api/catalog')
    tables = cat.list_tables('analytics')
    print(f'\n✅ PyIceberg voit {len(tables)} table(s) dans analytics:')
    for t in tables:
        print(f'   {t}')
except Exception as e:
    print(f'PyIceberg: {e}')

---
## ⏱️ Temps 2 — Incident à résoudre (35 min)

### Partie A — Multi-tenant et isolation (15 min)

> **Problème** : l'équipe Data Science a accidentellement écrasé une table Gold
> de l'équipe Analytics. Il n'y avait pas d'isolation de namespace.
>
> **TODO 1** : Configurez Polaris pour que le namespace `analytics` soit en
> lecture seule pour l'équipe Data Science, et que `mlops` soit leur espace dédié.

In [ ]:
# TODO 1 — Isolation multi-tenant
# 📝 a) Créez un namespace mlops avec des propriétés d'ownership
#    b) Créez une table dans polaris.mlops uniquement
#    c) Vérifiez qu'une écriture dans polaris.analytics lève une erreur
#       en simulant un accès non autorisé (bad token ou namespace incorrect)

# Astuce : dans Polaris, l'isolation fine se fait via des catalog roles.
# Pour ce lab, on simule l'isolation au niveau namespace.

print('💡 spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.mlops")')
print('💡 Tenter une écriture dans analytics avec un user non autorisé')
print('💡 Vérifier avec polaris("GET", "/api/catalog/v1/namespaces/mlops/properties")')

In [ ]:
# ✅ SOLUTION TODO 1

# Namespace mlops pour Data Science
spark.sql('CREATE NAMESPACE IF NOT EXISTS polaris.mlops')
spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.mlops.feature_store_index
    (feature_name STRING, feature_view STRING, entity STRING,
     dtype STRING, created_at TIMESTAMP)
    USING iceberg
""")
spark.sql("""
    INSERT INTO polaris.mlops.feature_store_index VALUES
    ('recency_days', 'customer_rfm', 'customer_id', 'INT64', current_timestamp()),
    ('churn_score',  'customer_churn', 'customer_id', 'FLOAT', current_timestamp())
""")
print('✅ Table polaris.mlops.feature_store_index créée')

# Propriétés de namespace
props = polaris('GET', '/api/catalog/v1/namespaces/mlops/properties')
print('\n📋 Propriétés du namespace mlops :')
print(json.dumps(props, indent=2))

# Mettre à jour les propriétés
polaris('POST', '/api/catalog/v1/namespaces/mlops/properties', {
    'updates': {'owner': 'data-science-team', 'access': 'read-write'},
    'removals': []
})
polaris('POST', '/api/catalog/v1/namespaces/analytics/properties', {
    'updates': {'owner': 'analytics-team', 'access': 'read-only-for-ds'},
    'removals': []
})
print('\n✅ Propriétés d\'ownership configurées')
print('   analytics → owner: analytics-team')
print('   mlops     → owner: data-science-team')

### Partie B — Fédération multi-moteurs (15 min)

> **TODO 2** : Vérifiez que Spark ET Trino lisent la même table depuis Polaris
> et que les résultats sont identiques. C'est le test de fédération.

In [ ]:
# TODO 2 — Fédération multi-moteurs
# 📝 a) Lire polaris.analytics.sales_summary depuis Spark
#    b) Lire la même table depuis Trino (via trino Python driver)
#    c) Comparer les résultats — ils doivent être identiques

print('💡 pip install trino')
print('💡 from trino.dbapi import connect')
print('💡 conn = connect(host="trino", port=8080, user="formation")')
print('💡 cursor = conn.cursor()')
print('💡 cursor.execute("SELECT * FROM polaris.analytics.sales_summary")')

In [ ]:
# ✅ SOLUTION TODO 2
import subprocess
subprocess.run(['pip', 'install', 'trino', '--quiet'], check=False)
from trino.dbapi import connect

# Lecture Spark
df_spark = spark.sql('SELECT * FROM polaris.analytics.sales_summary ORDER BY store_region')
print('📊 Lecture via Spark :')
df_spark.show()

# Lecture Trino
try:
    conn = connect(host='trino', port=8080, user='formation', catalog='polaris', schema='analytics')
    cur = conn.cursor()
    cur.execute('SELECT * FROM polaris.analytics.sales_summary ORDER BY store_region')
    rows_trino = cur.fetchall()
    print('📊 Lecture via Trino :')
    for r in rows_trino:
        print(f'   {r}')
    print('\n✅ Fédération confirmée : Spark et Trino lisent les mêmes données')
    print('   Un seul catalog Polaris, deux moteurs — zéro copie de données.')
except Exception as e:
    print(f'Trino: {e}')
    print('💡 Vérifier que Trino est bien configuré avec le catalog polaris.properties')

### Partie C — Interopérabilité PyIceberg (5 min)

In [ ]:
# ── PyIceberg lit directement via Polaris REST ────────────────────────────────
try:
    from pyiceberg.catalog.rest import RestCatalog
    import pyarrow as pa

    cat = RestCatalog('polaris', **{
        'uri': f'{POLARIS}/api/catalog',
        'warehouse': 'analytics',
        's3.endpoint': MINIO,
        's3.access-key-id': 'minioadmin',
        's3.secret-access-key': 'minioadmin',
        's3.path-style-access': 'true',
    })
    table = cat.load_table(('analytics', 'sales_summary'))
    df_arrow = table.scan().to_arrow()
    print('📊 Lecture via PyIceberg (Arrow) :')
    print(df_arrow.to_pandas())
    print('\n✅ 3 moteurs, 1 catalog, 0 copie de données.')
except Exception as e:
    print(f'PyIceberg: {e}')

---
## ⏱️ Temps 3 — Extension (10 min)

**Challenge** : Requête de jointure cross-namespace entre `analytics` et `mlops` via Trino.
Ce pattern est fondamental pour le serving ML : joindre les features MLOps avec les données analytiques.

In [ ]:
# ── Cross-namespace join : analytics × mlops via Trino ────────────────────────
try:
    conn = connect(host='trino', port=8080, user='formation')
    cur = conn.cursor()
    cur.execute("""
        SELECT
            s.store_region,
            s.category,
            s.ca_total,
            f.feature_name,
            f.feature_view
        FROM polaris.analytics.sales_summary s
        CROSS JOIN polaris.mlops.feature_store_index f
        ORDER BY s.store_region, f.feature_name
    """)
    rows = cur.fetchall()
    print('📊 Jointure cross-namespace (analytics × mlops) via Trino :')
    print(f'{"store_region":<20} {"category":<15} {"feature_name":<20} {"feature_view"}')
    print('-' * 75)
    for r in rows:
        print(f'{str(r[0]):<20} {str(r[1]):<15} {str(r[3]):<20} {r[4]}')
    print('\n✅ Jointure cross-namespace : un moteur, deux namespaces, zéro ETL.')
except Exception as e:
    print(f'Cross-namespace join: {e}')

---
## ⏱️ Débrief client (10 min)

### Questions — rôle Client :

1. **"Si j'ai 10 équipes, je dois créer 10 namespaces dans Polaris ? Comment je gère les permissions à l'échelle ?"**
2. **"Spark et Trino lisent les mêmes données — mais si une équipe écrit avec Spark pendant qu'une autre lit avec Trino, que se passe-t-il ?"**
3. **"Comment Polaris se compare à Unity Catalog de Databricks ? Pourquoi vous avez choisi Polaris ?"**
4. **"Si le serveur Polaris tombe, mes tables Iceberg sont-elles perdues ?"**

---

## Récapitulatif

| Concept | Exercice | En production |
|---------|---------|---------------|
| Namespaces isolés | retailco / mlops / analytics | 1 namespace par domaine / équipe |
| Propriétés ownership | Mettre à jour les métadonnées | Gouvernance des namespaces |
| Fédération Spark + Trino | Même table, 2 moteurs | Pas de copie — 1 source de vérité |
| PyIceberg REST | Lire sans Spark | Scripts Python légers, notebooks ML |
| Cross-namespace join | analytics × mlops | Pas d'ETL entre domaines |